In [1]:
from transformers import CLIPProcessor, CLIPModel
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import Imagenette
from sklearn.metrics import accuracy_score
import numpy as np
import torchvision
from tqdm import tqdm

/home/shkaf2m/Desktop/ml-isp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
model = model.half()
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [3]:
def custom_collate_function(batch):
  images = [item[0] for item in batch]
  labels = [item[1] for item in batch]
  labels = torch.tensor(labels, dtype=torch.long)
  return images, labels

In [4]:
val_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
])

val_dataset = Imagenette(root='./data', split='val', download=True, transform = val_transform)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=7, collate_fn = custom_collate_function)

In [5]:
def ValidateModel(model, device, data_loader, text_features):
  
  model.eval()
  true_labels = torch.tensor([]).to(device)
  predicted_labels = torch.tensor([]).to(device)

  text_features = text_features / text_features.norm(dim=-1, keepdim=True)
  logit_scale = model.logit_scale.exp()

  with torch.no_grad():
    for images, labels in tqdm(data_loader):
      labels = labels.to(device)
      image_inputs = processor(images = images, return_tensors = "pt", padding = True).to(device)
      image_features = model.get_image_features(**image_inputs)
       # Косинусное сходство между image features и text features + масштабирование
      similarity = (image_features @ text_features.T) * logit_scale

      predicted = similarity.argmax(dim=1)
      true_labels = torch.cat((true_labels, labels), 0)
      predicted_labels = torch.cat((predicted_labels, predicted), 0)

  return true_labels, predicted_labels

In [ ]:
import os
from sklearn.metrics import f1_score
os.environ["TOKENIZERS_PARALLELISM"] = "false"

all_classes = [label[0] for label in val_dataset.classes]
text_inputs = processor(text = all_classes, return_tensors = "pt", padding = True).to(DEVICE)

model = model.to(DEVICE)
model.eval()

text_features = model.get_text_features(**text_inputs)

Замерим используемые ресурсы без half

In [8]:

true_data, predicted_data = ValidateModel(model, device = DEVICE, data_loader = val_loader, text_features = text_features)
f1_res = f1_score(true_data.cpu().numpy(), predicted_data.cpu().numpy(), average = 'macro')

print("F1 Score: ", f1_res)

100%|██████████| 31/31 [01:42<00:00,  3.30s/it]


F1 Score:  0.9879248214523058


Без Half:

До измерений:

Температура: 49 градусов
Мощность: 4.38 Вт
Память: 769 MiB / 4096 MiB
Нагрузка: 7%



Во время измерений:

Температура: 65 градусов
Мощность: 50.11 Вт
Память: 1289 MiB / 4096 MiB
Нагрузка: 80%

При использовании half:

До измерений:

Температура: 45 градусов
Мощность: 5.39 Вт
Память: 447 MiB / 4096 MiB
Нагрузка: 11%


Во время измерений:
Температура: 52 градусов
Мощность: 32.41 Вт
Память: 821 MiB / 4096 MiB
Нагрузка: 95%